# Test dqm fonctionnalities

### Test de la métrique de représentativité

Test de la métrique de représentativité sur le dataset : `output/large_test_2m_test.parquet`


In [9]:
from dqm_ml.processors.pipeline import DuckDBRepresentativenessProcessor

config = {
    "input_path": "../output/samples.parquet",
    "columns": ["class_id", "image_size"],  # adaptez
    "bins": 20,
    "batch_size": 100000,   # chunks DuckDB si dataset très grand
    "memory_limit": "2GB",
    "threads": 4,
}
proc = DuckDBRepresentativenessProcessor(config)

# Pas besoin de compute_features(): DuckDB lit directement le fichier
results = proc.compute({})
results

{'class_id': {'basic_stats': {'count': 2100,
   'mean': 10.0,
   'std': 6.056742961569779,
   'variance': 36.66666666666671,
   'min': 0,
   'max': 20},
  'shannon-entropy': {'entropy': 4.183159450265486,
   'interpretation': 'high_diversity',
   'method': 'duckdb'},
  'grte': {'grte_value': 0.9678919589647886,
   'interpretation': 'high_representativeness',
   'method': 'duckdb'},
  'chi-square': {'p_value': 0.05,
   'chi_square_stat': 4.761904761904762,
   'interpretation': 'follows_distribution',
   'method': 'duckdb'}},
 'image_size': {'basic_stats': {'count': 2100,
   'mean': 209089.16190476192,
   'std': 1890.5189888342231,
   'variance': 3572360.11283461,
   'min': 194420,
   'max': 214344},
  'shannon-entropy': {'entropy': 2.8122529347694347,
   'interpretation': 'high_diversity',
   'method': 'duckdb'},
  'grte': {'grte_value': 0.6506940590002406,
   'interpretation': 'high_representativeness',
   'method': 'duckdb'},
  'chi-square': {'p_value': 0.001,
   'chi_square_stat': 49

#### test de scalabilité sur un dataset à 2 millions de rows

In [ ]:
from dqm_ml.processors.pipeline import DuckDBRepresentativenessProcessor

config = {
    "input_path": "../output/large_test_2m.parquet", 
    "columns": ["class_id", "brightness", "image_size"],
    "bins": 25,
    "batch_size": 100000,
    "memory_limit": "2GB",
    "threads": 4,
}

In [50]:
proc = DuckDBRepresentativenessProcessor(config)
results = proc.compute([]) # Beaucoup plus rapide que la version avec batching, mais gère moins bien les gros datasets
results
proc.close()

🦆 DuckDB: Agrégation des statistiques de 0 batch(es)
📊 Calcul des statistiques partielles...
📊 Calcul des stats partielles pour 'class_id'...
   ✅ Count: 2000000, Range: [0, 20], Sum: 19965130
📊 Calcul des stats partielles pour 'brightness'...
   ✅ Count: 2000000, Range: [0.05812249505585472, 255.0], Sum: 329178282.4042266
📊 Calcul des stats partielles pour 'image_size'...
   ✅ Count: 2000000, Range: [50000, 350598], Sum: 400049710895
📊 Agrégation des statistiques de 1 batch(es)
📊 Agrégation pour colonne 'class_id'...
   ✅ Count: 2,000,000, Mean: 9.983, Range: [0.000, 20.000]
📊 Agrégation pour colonne 'brightness'...
   ✅ Count: 2,000,000, Mean: 164.589, Range: [0.058, 255.000]
📊 Agrégation pour colonne 'image_size'...
   ✅ Count: 2,000,000, Mean: 200024.855, Range: [50000.000, 350598.000]
📊 Métriques DQM-ML pour 'class_id': 2,000,000 valeurs
   ✅ Shannon: 4.169130, GRTE: 0.897773
📊 Métriques DQM-ML pour 'brightness': 2,000,000 valeurs
   ✅ Shannon: 3.645069, GRTE: 0.784923
📊 Métriques

### Utilisation de la métrique de représentativité avec batching

In [52]:
import importlib

import dqm_ml.processors.pipeline as pipeline

importlib.reload(pipeline)

from dqm_ml.processors.pipeline import DuckDBRepresentativenessProcessor_batch

config = {
    "input_path": "../output/large_test_2m.parquet",
    "columns": ["class_id","brightness","image_size"],
    "bins": 25,
    "batch_size": 100000,
    "progress": True,          # important
    "progress_every": 1,       # logs à chaque batch
}

proc = DuckDBRepresentativenessProcessor_batch(config)
results = proc.compute()       # ne pas passer de batch_metrics
proc.close()

# Accéder sans KeyError
for log in results.get("batch_progress", [])[:5]:
    print(log)

📦 Batch 1: 100,000 lignes, 0.061s, prog 5.0%
📦 Batch 2: 100,000 lignes, 0.061s, prog 10.0%
📦 Batch 3: 100,000 lignes, 0.062s, prog 15.0%
📦 Batch 4: 100,000 lignes, 0.063s, prog 20.0%
📦 Batch 5: 100,000 lignes, 0.072s, prog 25.0%
📦 Batch 6: 100,000 lignes, 0.073s, prog 30.0%
📦 Batch 7: 100,000 lignes, 0.077s, prog 35.0%
📦 Batch 8: 100,000 lignes, 0.081s, prog 40.0%
📦 Batch 9: 100,000 lignes, 0.086s, prog 45.0%
📦 Batch 10: 100,000 lignes, 0.086s, prog 50.0%
📦 Batch 11: 100,000 lignes, 0.084s, prog 55.0%
📦 Batch 12: 100,000 lignes, 0.087s, prog 60.0%
📦 Batch 13: 100,000 lignes, 0.089s, prog 65.0%
📦 Batch 14: 100,000 lignes, 0.093s, prog 70.0%
📦 Batch 15: 100,000 lignes, 0.101s, prog 75.0%
📦 Batch 16: 100,000 lignes, 0.103s, prog 80.0%
📦 Batch 17: 100,000 lignes, 0.124s, prog 85.0%
📦 Batch 18: 100,000 lignes, 0.120s, prog 90.0%
📦 Batch 19: 100,000 lignes, 0.113s, prog 95.0%
📦 Batch 20: 100,000 lignes, 0.121s, prog 100.0%
{'batch_index': 1, 'offset': 0, 'limit': 100000, 'duration_s': 0.0606

In [53]:
results

{'class_id': {'shannon-entropy': {'entropy': 4.372831486404697,
   'interpretation': 'high_diversity',
   'method': 'duckdb_batch_combined'},
  'grte': {'grte_value': 0.9416380067998671,
   'interpretation': 'high_representativeness',
   'method': 'duckdb_batch_combined'},
  'chi-square': {'p_value': 0.001,
   'chi_square_stat': 448344.755325,
   'interpretation': 'does_not_follow_distribution',
   'method': 'duckdb_batch_combined'},
  'basic_stats': {'count': 2000000,
   'sum': 19965130.0,
   'mean': 9.982565,
   'std': 6.015581187281493,
   'variance': 36.18721702077501,
   'min': 0.0,
   'max': 20.0}},
 'brightness': {'shannon-entropy': {'entropy': 4.041339352459938,
   'interpretation': 'high_diversity',
   'method': 'duckdb_batch_combined'},
  'grte': {'grte_value': 0.8702550611620007,
   'interpretation': 'high_representativeness',
   'method': 'duckdb_batch_combined'},
  'chi-square': {'p_value': 0.001,
   'chi_square_stat': 3038868.0722749997,
   'interpretation': 'does_not_fol